In [12]:
from pathlib import Path

import ray
import torch

from climanet.tune import run_tune
from climanet.dataset import DatasetConfig, DataLoaderConfig
from climanet.predict import predict_monthly_var, PredictionConfig

import xarray as xr

In [2]:
data_folder = Path("./eso4clima/dc_data")
run_dir = Path("./runs_daily").resolve()

var_name = "tos"

daily_data = xr.open_mfdataset(data_folder / f"202101_day_ERA5dc_masked_{var_name}.nc")
daily_data_validation = xr.open_mfdataset(data_folder / f"202102_day_ERA5dc_masked_{var_name}.nc")
daily_data_test = xr.open_mfdataset(data_folder / f"202103_day_ERA5dc_masked_{var_name}.nc")

monthly_data = xr.open_mfdataset(data_folder / f"202101_mon_ERA5dc_full_{var_name}.nc")
monthly_data_validation = xr.open_mfdataset(data_folder / f"202102_mon_ERA5dc_full_{var_name}.nc")
monthly_data_test = xr.open_mfdataset(data_folder / f"202103_mon_ERA5dc_full_{var_name}.nc")

file_name = data_folder / "era5_lsm_bool.nc"  # downloded from era5 and regridded using the function `regrid_to_boundary_centered_grid`
lsm_mask = xr.open_dataset(file_name)

### prepare data for tuning

In [3]:
# coordinates of subset
lon_subset = slice(-50, 50)  # one lon -179.9 is nan, check data
lat_subset = slice(-30, 10)

daily_subset = daily_data.sel(lon=lon_subset, lat=lat_subset)
monthly_subset = monthly_data.sel(lon=lon_subset, lat=lat_subset)
lsm_subset = lsm_mask.sel(lon=lon_subset, lat=lat_subset)  # True=Land

daily_validation_subset = daily_data_validation.sel(lon=lon_subset, lat=lat_subset)
monthly_validation_subset = monthly_data_validation.sel(lon=lon_subset, lat=lat_subset)

daily_test_subset = daily_data_test.sel(lon=lon_subset, lat=lat_subset)
monthly_test_subset = monthly_data_test.sel(lon=lon_subset, lat=lat_subset)

print(daily_subset[var_name].shape, monthly_subset[var_name].shape)  # (time, lat, lon)

(31, 160, 400) (1, 160, 400)


### config for hyper parameter tuning

In [4]:
data_config_train = {
    "input_data": ray.put(daily_subset),
    "monthly_data": ray.put(monthly_subset),
    "land_mask_data": ray.put(lsm_subset["lsm"]),
}

data_config_validation = {
    "input_data": ray.put(daily_validation_subset),
    "monthly_data": ray.put(monthly_validation_subset),
    "land_mask_data": ray.put(lsm_subset["lsm"]),
}

# dont use ray.put() (i.e. object store) when data is large
static_args = {
    "data_config_train": data_config_train,
    "data_config_validation": data_config_validation,
    "is_hourly": False,
    "var_name": "tos",
    "max_num_epochs": 1,
    "num_trials": 1,  # this is num_samples in ray.tune.TuneConfig
    "cpu_per_trial": 2,
    "gpu_per_trial": 0,
    "run_dir": run_dir,
    "device": "cpu",
    "dataloader_num_workers": 1,
    "dataset_patch_size": (1, 40, 40),
    "dataset_stride": (20, 20),
    "num_epoch": 1,
    "max_concurrent_trials": 1,  # less than GPUs per node (4) avoid OOM
    "experiment_name": "climanet_tune",
}

# parameters to tune
tune_config = {
    "patch_size": ray.tune.grid_search([2, 4]),
    "overlap": 2,
    "embed_dim": 32,
    "dropout": 0.0,
    "hidden": 32,
    "spatial_depth": 3,
    "spatial_heads": 4,
    "optimizer_lr": 1e-1,
    "batch_config": {"batch_size": 10, "accumulation_steps": 5},
}

2026-08-03 19:54:40,334	INFO worker.py:2024 -- Started a local Ray instance.
(_train pid=650516) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/climanet_tune/_train_6819a_00000_0_patch_size=2_2026-08-03_19-54-43/checkpoint_000000)


### Run ray tune

In [5]:
results = run_tune(tune_config, static_args)

ray.shutdown()

2026-08-03 19:55:10,670	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/climanet_tune' in 0.0048s.
2026-08-03 19:55:10,676	INFO tune.py:1039 -- Total run time: 27.56 seconds (27.53 seconds for the tuning loop).


### Inspect the output

In [6]:
results.get_best_result("loss", "min")

Result(
  metrics={'loss': 0.12806640352521623},
  path='/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/climanet_tune/_train_6819a_00001_1_patch_size=4_2026-08-03_19-54-56',
  filesystem='local',
  checkpoint=Checkpoint(filesystem=local, path=/home/sarah/GitHub/ClimaNet/notebooks/runs_daily/climanet_tune/_train_6819a_00001_1_patch_size=4_2026-08-03_19-54-56/checkpoint_000000)
)

### Check best model

In [7]:
# find the path to best model
if not ray.is_initialized():
    ray.init()

experiment_path = results.experiment_path  # or add the path above manually as Path("./runs_daily/_train_2026-07-23_10-11-56\").resolve()"
analysis = ray.tune.ExperimentAnalysis(experiment_path)
best_result = analysis.get_best_trial("loss", "min")
best_checkpoint = best_result.checkpoint
model_path = Path(best_checkpoint.path) / "checkpoint.pt"

2026-08-03 19:55:34,568	INFO worker.py:2024 -- Started a local Ray instance.


In [16]:
dataset_config = DatasetConfig(
    is_hourly=static_args["is_hourly"],
    var_name=static_args["var_name"],
    spatial_dims=("lat", "lon"),
    patch_size=static_args["dataset_patch_size"],
    stride=static_args["dataset_stride"],
    sh_pos_table=None,
    sh_embed_dim=96,
    sh_order_L=10,
    verbose=False,
)

use_cuda = static_args["device"] == "cuda"
dataloader_config = DataLoaderConfig(
    batch_size=tune_config["batch_config"]["batch_size"],
    shuffle=True,
    num_workers=static_args["dataloader_num_workers"],
    pin_memory=use_cuda,
    persistent_workers=False,
    device=static_args["device"],
)

prediction_config = PredictionConfig(
    calculate_residuals=True,
    return_numpy=False,
    save_predictions=False,
    return_loss=True,
    device=static_args["device"],
    verbose=True,
)

In [17]:
# inference on test data
test_loss = predict_monthly_var(
    model=model_path,
    input_data=daily_test_subset, 
    monthly_data=monthly_test_subset, 
    dataset_config=dataset_config,
    dataloader_config=dataloader_config,
    prediction_config=prediction_config,
    land_mask=lsm_subset["lsm"],
    run_dir=run_dir,
)

Processed batch 14, with average loss: 0.1287
Average loss over all batches: 0.1287


In [18]:
test_loss

0.12866921084267752